In [5]:
from pathlib import Path

filepath = "input.txt"

with open(filepath,"r", encoding="utf-8") as f:
    text = f.read()

In [6]:
print(text[:200])
print("Length of dataset in characters : ",len(text))

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you
Length of dataset in characters :  1115394


In [7]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)



 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [8]:
stoi = { ch:i for i,ch in enumerate(chars)}     #it gives index to all the characters ( unique characters )
itos = { i:ch for i,ch in enumerate(chars)}     #it retrieve the index and checks the acctual value of that index
encode = lambda s: [stoi[c] for c in s]         #converts the input string to there character index values means string -> integer
decode = lambda l: ''.join([itos[i] for i in l])        #convers the integer -> string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [9]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape,data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [10]:
n = int(0.9 * len(data)) #training data 90%
train_data = data[:n]
val_data = data[n:]

In [11]:
blocksize = 8
train_data[:blocksize + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [12]:
x_train = train_data[:blocksize]
y_train = train_data[1:blocksize+1]

for t in range(blocksize):
    context = x_train[:t+1]
    target = y_train[t]
    print(f"when the input is {context} then the target {target}")

when the input is tensor([18]) then the target 47
when the input is tensor([18, 47]) then the target 56
when the input is tensor([18, 47, 56]) then the target 57
when the input is tensor([18, 47, 56, 57]) then the target 58
when the input is tensor([18, 47, 56, 57, 58]) then the target 1
when the input is tensor([18, 47, 56, 57, 58,  1]) then the target 15
when the input is tensor([18, 47, 56, 57, 58,  1, 15]) then the target 47
when the input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) then the target 58


In [40]:
torch.manual_seed(1337)
#kinda matrix like 4 rows and 8 columns (logic - random values )
batch_size = 4
block_size = 8


def get_batch(split):
    #generate small batches of data ......
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y


xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('-----------')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b,:t+1]
        target = yb[b,t]
        print(f"When input is {context} the target is {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
-----------
When input is tensor([24]) the target is 43
When input is tensor([24, 43]) the target is 58
When input is tensor([24, 43, 58]) the target is 5
When input is tensor([24, 43, 58,  5]) the target is 57
When input is tensor([24, 43, 58,  5, 57]) the target is 1
When input is tensor([24, 43, 58,  5, 57,  1]) the target is 46
When input is tensor([24, 43, 58,  5, 57,  1, 46]) the target is 43
When input is tensor([24, 43, 58,  5, 57,  1, 46, 43]) the target is 39
When input is tensor([44]) the target is 53
When input is tensor([44, 53]) the target is 56
When input is tensor([44, 53, 56]) the ta

In [38]:
import torch
import torch.nn as nn   
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(
            vocab_size,
            vocab_size
        )

    def forward(self, idx, targets=None):

        logits = self.token_embedding_table(idx)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape

            logits = logits.view(B*T, C)
            targets = targets.view(B*T) # or we can use -1  (to flattern)
            
            loss = F.cross_entropy(logits, targets)
        return logits,loss
    
    def generate(self,idx,max_new_tokens):
        #idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            #get the predictions
            logits, loss = self(idx)
            #focus only on the last time step
            logits = logits[:,-1,:] #becomes B, C
            #apply softmax to predictions
            probs = F.softmax(logits,dim=1) # B, C
            #sample from the distribution
            idx_next = torch.multinomial(probs, num_samples = 1) #B, 1
            #append sampled index to running sequence
            idx = torch.cat((idx, idx_next), dim=1) #B, T+1
        return idx
            


model = BigramLanguageModel(vocab_size)
logits,loss = model(xb,yb)
print(logits.shape)
print(loss)
print(decode(model.generate(idx = torch.zeros((1,1),dtype=torch.long), max_new_tokens=100)[0].tolist()))
print(len(decode(model.generate(idx = torch.zeros((1,1),dtype=torch.long), max_new_tokens=100)[0].tolist())))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ
101


In [58]:
eval_iters = 200
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train','eval']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X , Y = get_batch(split)
            logits, loss = model(X,Y)
            losses = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

------- START TRAINING THE MODEL -------

In [59]:
#create pytorch optimizer
optimizer = torch.optim.AdamW(model.parameters(),lr=3)
batch_size = 32

for iter in range(100):
    
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter} Train loss: {losses['train']:.4f} Val loss: {losses['eval']:.4f}")
    #sample data
    xb,yb = get_batch('train')
    
    logits, loss = model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
print(loss.item())

AttributeError: 'float' object has no attribute 'mean'

In [48]:
print(decode(model.generate(idx = torch.zeros((1,1),dtype=torch.long), max_new_tokens=400)[0].tolist()))


BAd iglithouglearovel:
Trey f owr w, f; CEThevith
Tharthew, ggow, brathes far pmevatopm pmoerer vefief ou orofaritithicathecopme ENIV: me, Pl'dve br'dleleeve, pmowithevarearoffankerthemithely,
owesswer w, ind gorecothar; pere avar fare fascoeaththoliglld gly, l:
Thethaike it gichepmotithe, y mbe alitil:
Morokelwsthoneme hterind bead!
Th m.
Th IO:
oth ENoe, owrrar glig alige depmbracke, of t mus.
I
